# Full candidate M0 audit — fresh Qwen3-1.7B

Before scaling M1, measure the untrained pinned 1.7B base on all 180 candidate units: 30 F +90 R +60 new relational units, seven questions each. Use only m1_dev and n_rel_train: 1,260 questions in each declared mode, **2,520 predictions, zero training updates, no adapter loaded and no final-test generation**.

Keep the current candidate unchanged. The static acceptance review found surface overlaps, including a real book-title collision for The Silver Citadel; those records are flagged, not silently renamed/filtered. Audit completion or zero matches do not accept the dataset or certify absent internal knowledge. Review actual matches/mentions, format/cap issues and collision context before full training.

Use the existing Kaggle GPU kernel. Publish this runner/config/review metadata first. HF_TOKEN is optional for public-model downloads; no new checkpoint or model upload is performed. Download the final evidence ZIP.


In [1]:
import os, subprocess, sys, tempfile
from pathlib import Path

remote_root = Path('/kaggle/working')
assert remote_root.is_dir(), 'Connect to the Kaggle GPU kernel.'
repo_dir = Path(tempfile.mkdtemp(prefix='full-m0-qwen17-source-', dir=remote_root)) / 'repo'
subprocess.run(['git', 'clone', '--branch', 'main', 'https://github.com/DevulapalliTharun/persistent-llm-unlearning.git', str(repo_dir)], check=True)
os.chdir(repo_dir)
print('Repository commit:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())
assert Path('src/evaluation/full_m0.py').is_file(), 'Publish the full M0 runner/config/review files first.'
assert Path('configs/evaluation/m0_qwen17_full_v1.json').is_file()


Cloning into '/kaggle/working/full-m0-qwen17-source-u4_51bpc/repo'...


Repository commit: 667a2b836091be7a4c3c7cc4c5a6ae3d9a64b3b0


In [2]:
assert Path('/kaggle/working').is_dir()
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '-r', 'requirements/m1-tiny.txt'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'], check=True)
subprocess.run([sys.executable, '-c', 'import torch, transformers; assert torch.cuda.is_available(); print(torch.__version__, transformers.__version__)'], check=True)


Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
2.11.0+cu130 5.20.2


CompletedProcess(args=['/usr/bin/python3', '-c', 'import torch, transformers; assert torch.cuda.is_available(); print(torch.__version__, transformers.__version__)'], returncode=0)

In [3]:
import json

plan_dir = Path(tempfile.mkdtemp(prefix='full-m0-qwen17-plan-', dir=remote_root))
subprocess.run([sys.executable, '-m', 'src.evaluation.full_m0', '--config', 'configs/evaluation/m0_qwen17_full_v1.json', '--output-dir', str(plan_dir), '--mode', 'plan'], check=True)
plan = json.loads((plan_dir / 'plan.json').read_text())
assert plan['planned_predictions'] == 2520 and plan['knowledge_units'] == 180
assert plan['training_updates'] == 0 and not plan['adapters_loaded'] and not plan['dataset_accepted']
print('Non-final inputs/checksums and frozen review reports verified.')
print('Counts per mode:', plan['by_entity_partition_per_mode'])
print('External surface-review record IDs:', plan['externally_flagged_record_ids'])


{
  "config": {
    "protocol_version": "answer-only-v2",
    "model_id": "Qwen/Qwen3-1.7B",
    "model_revision": "70d244cc86ccca08cf5af4e1e306ecf908b1ad5e",
    "bundle": "data/candidates/pilot-v1/seed-42",
    "expected_checksum_index_sha256": "1a70624618acb6a849f1605766a79eef0f66f8b0aab938ed500152d1705ebe05",
    "prompt_files": [
      "m1_dev.jsonl",
      "n_rel_train.jsonl"
    ],
    "batch_size": 16,
    "seed": 42,
    "system_prompt": "Answer the question using one line: Answer: <answer>. Replace <answer> with only the requested author name, book title, prize name, year, or integer. If you do not know, reply Answer: Unknown. Do not explain, repeat the question, or list alternatives.",
    "run_type": "qwen17_full_m0_dual_mode_v1",
    "answer_only_max_new_tokens": 128,
    "thinking_max_new_tokens": 1024,
    "sampling": {
      "do_sample": true,
      "temperature": 0.6,
      "top_p": 0.95,
      "top_k": 20,
      "min_p": 0.0
    },
    "structural_review": "experiment

In [4]:
# Fresh public base only: no tiny M1 adapter or private checkpoint restoration.
run_env = os.environ.copy()
try:
    from kaggle_secrets import UserSecretsClient
    download_token = UserSecretsClient().get_secret('HF_TOKEN')
except Exception:
    download_token = None
if download_token:
    run_env['HF_TOKEN'] = download_token
output_dir = Path(tempfile.mkdtemp(prefix='m0-qwen17-full-', dir=remote_root))
print('Audit outputs:', output_dir, flush=True)
try:
    subprocess.run([sys.executable, '-u', '-m', 'src.evaluation.full_m0', '--config', 'configs/evaluation/m0_qwen17_full_v1.json', '--output-dir', str(output_dir), '--mode', 'audit'], env=run_env, check=True)
finally:
    run_env.pop('HF_TOKEN', None)


Audit outputs: /kaggle/working/m0-qwen17-full-55wpqjfh


Loading weights: 100%|██████████| 311/311 [00:01<00:00, 159.38it/s]
[transformers] The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


answer_only/m1_dev: 16/840; 16/1260 mode predictions
answer_only/m1_dev: 32/840; 32/1260 mode predictions
answer_only/m1_dev: 48/840; 48/1260 mode predictions
answer_only/m1_dev: 64/840; 64/1260 mode predictions
answer_only/m1_dev: 80/840; 80/1260 mode predictions
answer_only/m1_dev: 96/840; 96/1260 mode predictions
answer_only/m1_dev: 112/840; 112/1260 mode predictions
answer_only/m1_dev: 128/840; 128/1260 mode predictions
answer_only/m1_dev: 144/840; 144/1260 mode predictions
answer_only/m1_dev: 160/840; 160/1260 mode predictions
answer_only/m1_dev: 176/840; 176/1260 mode predictions
answer_only/m1_dev: 192/840; 192/1260 mode predictions
answer_only/m1_dev: 208/840; 208/1260 mode predictions
answer_only/m1_dev: 224/840; 224/1260 mode predictions
answer_only/m1_dev: 240/840; 240/1260 mode predictions
answer_only/m1_dev: 256/840; 256/1260 mode predictions
answer_only/m1_dev: 272/840; 272/1260 mode predictions
answer_only/m1_dev: 288/840; 288/1260 mode predictions
answer_only/m1_dev: 30

In [5]:
summary = json.loads((output_dir / 'summary.json').read_text())
assert summary['execution_complete'] and summary['training_updates'] == 0
assert not summary['adapters_loaded'] and not summary['weights_updated'] and not summary['dataset_accepted']
for mode, data in summary['modes'].items():
    score = data['answer_scoring']
    print(mode, '| answer matches:', score['answer_matches'], '/1260', '| format:', score['format_violations'], '| caps:', score['token_limit_hits'], '| missing boundaries:', data['missing_thinking_boundaries'], '| trace mentions:', data['reasoning_answer_mentions'], '| priority answer-review rows:', data['potential_answer_review_rows'])
print('Review files: potential_answers_for_review.json, protocol_issues_for_review.json, external_collision_rows_for_review.json')
print('All raw rows remain in the prediction files. Review findings before dataset acceptance; full M1 training is not authorized by this execution flag.')


answer_only | answer matches: 9 /1260 | format: 809 | caps: 0 | missing boundaries: 0 | trace mentions: 0 | priority answer-review rows: 9
thinking | answer matches: 1 /1260 | format: 752 | caps: 16 | missing boundaries: 15 | trace mentions: 4 | priority answer-review rows: 4
Review files: potential_answers_for_review.json, protocol_issues_for_review.json, external_collision_rows_for_review.json
All raw rows remain in the prediction files. Review findings before dataset acceptance; full M1 training is not authorized by this execution flag.


In [6]:
import zipfile

archive = Path(str(output_dir) + '-evidence.zip')
with zipfile.ZipFile(archive, 'x', compression=zipfile.ZIP_DEFLATED) as bundle:
    for file in sorted(output_dir.iterdir()):
        if file.is_file() and file.suffix in {'.json', '.jsonl', '.txt'}:
            bundle.write(file, arcname=file.name)
print('Download from Kaggle Output:', archive)


Download from Kaggle Output: /kaggle/working/m0-qwen17-full-55wpqjfh-evidence.zip
